**Notebook 01: Sub-ICB prescribing data for the three West Midlands ICBs**

This notebook prepares the monthly ICB-level data used in the interrupted time series (notebook 02) and in the checks in notebook 05. It takes the national OpenPrescribing file for the dapagliflozin measure, keeps the three study ICBs for January 2025 to June 2026, checks the data, and flags the months from September 2025, the first full month of generic supply.

Inputs: `OpenPrescribing_Data/dapagliflozin_measure_sicbl.csv` and `QOF_2024_25/MAPPING_NHS_GEOGRAPHIES_2425.csv`.

Output: `op_wm_cleaned.csv` (54 rows, 3 ICBs x 18 months).

In this file the denominator is all SGLT2 items and the numerator is the SGLT2 items that are NOT generic dapagliflozin. Later notebooks turn this into the generic share.

**Step 1: Load libraries**

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


**Step 2: Load the national sub-ICB file**

OpenPrescribing calls these organisations `ccg` in the `org_type` column, although they are now sub-ICB locations. The file covers every sub-ICB in England and runs past the study window, so the first look checks its size and date range.

In [2]:
op_sicbl = pd.read_csv('OpenPrescribing_Data/dapagliflozin_measure_sicbl.csv', parse_dates=['date'])

print('Shape:', op_sicbl.shape)
print('Columns:', op_sicbl.columns.tolist())
print('Measure:', op_sicbl['measure'].unique())
print('Organisation type:', op_sicbl['org_type'].unique())
print('Sub-ICBs in the file:', op_sicbl['org_id'].nunique())
print('Date range:', op_sicbl['date'].min().date(), 'to', op_sicbl['date'].max().date())

Shape: (6466, 9)
Columns: ['measure', 'org_type', 'org_id', 'org_name', 'date', 'numerator', 'denominator', 'calc_value', 'percentile']
Measure: <ArrowStringArray>
['dapagliflozin']
Length: 1, dtype: str
Organisation type: <ArrowStringArray>
['ccg']
Length: 1, dtype: str
Sub-ICBs in the file: 106
Date range: 2021-07-01 to 2026-07-01


**Step 3: Find the three study ICBs and confirm their codes**

The search lists every sub-ICB with a name linked to the West Midlands area. Only three are used in the study: Birmingham and Solihull (15E), Black Country (D2P2L) and Coventry and Warwickshire (B2M3M). The others in the list (the Staffordshire, Herefordshire and Worcestershire, and Shropshire areas) are in the wider Midlands but are outside the study. The second table checks the three codes against the QOF mapping file, which also gives the ICB codes (QHL, QUA and QWU) used in notebook 03.

In [3]:
pattern = ('BIRMINGHAM|SOLIHULL|BLACK COUNTRY|COVENTRY|WARWICK|WOLVERHAMPTON|'
           'STAFFORDSHIRE|WORCESTER|HEREFORD|SHROPSHIRE|TELFORD')
matches = op_sicbl[op_sicbl['org_name'].str.contains(pattern, case=False, na=False)][
    ['org_id', 'org_name']].drop_duplicates()
print('Sub-ICBs with a West Midlands related name:')
print(matches.to_string(index=False))

STUDY_CODES = ['15E', 'D2P2L', 'B2M3M']

geo = pd.read_csv('QOF_2024_25/MAPPING_NHS_GEOGRAPHIES_2425.csv')
print('\nStudy codes in the QOF mapping file:')
print(geo[geo['SUB_ICB_ODS_CODE'].isin(STUDY_CODES)]
      [['SUB_ICB_ODS_CODE', 'SUB_ICB_NAME', 'ICB_ODS_CODE', 'ICB_NAME']]
      .drop_duplicates().to_string(index=False))

Sub-ICBs with a West Midlands related name:
org_id                                           org_name
   15E                        NHS BIRMINGHAM AND SOLIHULL
   05G                            NHS NORTH STAFFORDSHIRE
   05D                             NHS EAST STAFFORDSHIRE
 D2P2L                                  NHS BLACK COUNTRY
   05Q NHS SOUTH EAST STAFFORDSHIRE AND SEISDON PENINSULA
 B2M3M                      NHS COVENTRY AND WARWICKSHIRE
 M2L0M                 NHS SHROPSHIRE, TELFORD AND WREKIN
   18C               NHS HEREFORDSHIRE AND WORCESTERSHIRE

Study codes in the QOF mapping file:
SUB_ICB_ODS_CODE                              SUB_ICB_NAME ICB_ODS_CODE                                            ICB_NAME
           B2M3M NHS Coventry and Warwickshire ICB - B2M3M          QWU NHS Coventry and Warwickshire Integrated Care Board
             15E     NHS Birmingham and Solihull ICB - 15E          QHL   NHS Birmingham and Solihull Integrated Care Board
           D2P2L        

**Step 4: Keep the three ICBs and the study period**

The study period is January 2025 to June 2026, which is 18 months. Rows before and after it are removed.

In [4]:
START, END = '2025-01-01', '2026-06-01'

three_icbs = op_sicbl[op_sicbl['org_id'].isin(STUDY_CODES)]
op_wm = three_icbs[(three_icbs['date'] >= START) & (three_icbs['date'] <= END)].copy()
op_wm = op_wm.sort_values(['org_id', 'date']).reset_index(drop=True)

print('Rows for the three ICBs, all dates:', len(three_icbs))
print('Rows in the study period:', len(op_wm))
print('Rows removed (outside the period):', len(three_icbs) - len(op_wm))
print(op_wm.groupby(['org_id', 'org_name']).size())

Rows for the three ICBs, all dates: 183
Rows in the study period: 54
Rows removed (outside the period): 129
org_id  org_name                     
15E     NHS BIRMINGHAM AND SOLIHULL      18
B2M3M   NHS COVENTRY AND WARWICKSHIRE    18
D2P2L   NHS BLACK COUNTRY                18
dtype: int64


**Step 5: Data quality checks**

The checks cover missing values, duplicate rows, gaps in the monthly sequence and whether the numbers are consistent. Each ICB should have all 18 months. The numerator should never be larger than the denominator, and `calc_value` should equal the numerator divided by the denominator.

In [5]:
print('Missing values:')
print(op_wm.isnull().sum().to_string())
print('\nDuplicate ICB-month rows:', op_wm.duplicated(['org_id', 'date']).sum())

expected_months = pd.date_range(START, END, freq='MS')
print('\nMonths per ICB (expected', len(expected_months), '):')
for org_id, d in op_wm.groupby('org_id'):
    missing = expected_months.difference(d['date'])
    print(' ', org_id, '| months:', len(d), '| missing:', list(missing.strftime('%b %Y')))

print('\nRows with zero denominator:', (op_wm['denominator'] <= 0).sum())
print('Rows with numerator above denominator:', (op_wm['numerator'] > op_wm['denominator']).sum())
print('calc_value equals numerator / denominator:',
      np.allclose(op_wm['calc_value'], op_wm['numerator'] / op_wm['denominator']))

print('\nMonthly SGLT2 items (denominator) by ICB:')
print(op_wm.groupby('org_name')['denominator'].describe().round(0).to_string())

Missing values:
measure        0
org_type       0
org_id         0
org_name       0
date           0
numerator      0
denominator    0
calc_value     0
percentile     0

Duplicate ICB-month rows: 0

Months per ICB (expected 18 ):
  15E | months: 18 | missing: []
  B2M3M | months: 18 | missing: []
  D2P2L | months: 18 | missing: []

Rows with zero denominator: 0
Rows with numerator above denominator: 0
calc_value equals numerator / denominator: True

Monthly SGLT2 items (denominator) by ICB:
                               count     mean     std      min      25%      50%      75%      max
org_name                                                                                          
NHS BIRMINGHAM AND SOLIHULL     18.0  31506.0  2706.0  26085.0  29384.0  31644.0  33419.0  35971.0
NHS BLACK COUNTRY               18.0  30980.0  2793.0  25690.0  28682.0  31176.0  33135.0  36247.0
NHS COVENTRY AND WARWICKSHIRE   18.0  26317.0  2183.0  21717.0  24922.0  26378.0  28077.0  29619.0


**Step 6: Flag the months from September 2025**

`post_patent` is 0 up to August 2025 and 1 from September 2025. The courts found the patent invalid in April and July 2025, generic products reached the market in early August 2025, and September 2025 was the first full month of generic supply. The column keeps the name post_patent so later notebooks still read it. Each ICB should have 8 months before and 10 after.

In [6]:
op_wm['post_patent'] = (op_wm['date'] >= '2025-09-01').astype(int)

print(op_wm.groupby('org_id')['post_patent'].value_counts().unstack().rename(
    columns={0: 'before', 1: 'after'}))

post_patent  before  after
org_id                    
15E               8     10
B2M3M             8     10
D2P2L             8     10


**Step 7: A first look at the generic share**

The measure counts items that are not generic dapagliflozin, so the generic share is 1 minus `calc_value`. The table shows it at five key months. It is only a quick check and is not saved. Notebook 02 builds the full time series.

In [7]:
view = op_wm.assign(generic_share_pct=(1 - op_wm['calc_value']) * 100)
key_months = pd.to_datetime(['2025-01-01', '2025-08-01', '2025-09-01', '2026-03-01', '2026-06-01'])
table = (view[view['date'].isin(key_months)]
         .pivot(index='org_name', columns='date', values='generic_share_pct'))
table.columns = table.columns.strftime('%b %Y')
print(table.round(1).to_string())

date                           Jan 2025  Aug 2025  Sep 2025  Mar 2026  Jun 2026
org_name                                                                       
NHS BIRMINGHAM AND SOLIHULL        53.0      56.4      56.4      57.8      69.7
NHS BLACK COUNTRY                  64.6      65.7      65.6      67.5      76.8
NHS COVENTRY AND WARWICKSHIRE      60.8      62.2      62.4      64.1      65.3


**Step 8: Save the cleaned file**

The file keeps the same ten columns as before, so notebooks 02 and 05 read it without changes.

In [8]:
columns = ['measure', 'org_type', 'org_id', 'org_name', 'date', 'numerator',
           'denominator', 'calc_value', 'percentile', 'post_patent']
op_wm[columns].to_csv('op_wm_cleaned.csv', index=False)

print('Saved op_wm_cleaned.csv:', op_wm[columns].shape)
print(op_wm.groupby('org_id')['date'].agg(['count', 'min', 'max']))

Saved op_wm_cleaned.csv: (54, 10)
        count        min        max
org_id                             
15E        18 2025-01-01 2026-06-01
B2M3M      18 2025-01-01 2026-06-01
D2P2L      18 2025-01-01 2026-06-01


**Acknowledgement of AI assistance**

This notebook was written by Mifta Mahmood with assistance from Claude (Anthropic), a generative AI assistant, which helped to write and debug the code. All code was run, checked and verified by the author, and the outputs shown are from the author's own runs. The final decisions on data, methods and interpretation are the author's own.